# 04a — AWS Braket QAOA Experiments (Scenario A, SV1)

This notebook runs a **Scenario A** QUBO on **AWS Braket (Amazon SV1)** using a simple, reproducible **p=1 QAOA** workflow:

- Load a pre-built Scenario A QUBO artifact from the repo (created in Phase 3).
- Upload the compact QUBO + an index→trial-id mapping to S3 (so the run is auditable).
- Execute QAOA circuits on **SV1** across a small (γ, β) grid.
- Decode the best measured bitstring back into selected trial IDs and compute objective metrics.
- Save results locally under `data/results/` and mirror them to S3 under a Braket results prefix.

Notes:
- Braket does **not** process the 557k XML corpus directly. The large S3 corpus is used upstream to engineer features and reduce to a **scenario candidate set**, which is then compressed to a **QUBO**. Braket runs the optimization on that compact QUBO instance.
- This notebook is intended to be “portfolio-clean”: deterministic mapping, explicit S3 prefixes, and saved artifacts.


In [3]:
# ============================================================
# Cell 1 — Setup: imports, paths, and output directories
# ============================================================

from pathlib import Path
import json
import math
import time
import numpy as np
import pandas as pd

import boto3

# Braket SDK imports (fail fast with a clear message if missing)
try:
    from braket.aws import AwsDevice
    from braket.circuits import Circuit
except Exception as e:
    raise ImportError(
        "Braket SDK not available in this environment.\n"
        "Fix (conda/pip): pip install amazon-braket-sdk\n"
        f"Underlying error: {e}"
    )

# --- Project directories ---
RESULTS_DIR = Path("data/results")
QUBO_DIR = Path("data/qubo_scenarios")
SCENARIO_DIR = Path("data/scenarios")
FIG_DIR = Path("outputs/figures")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
QUBO_DIR.mkdir(parents=True, exist_ok=True)
SCENARIO_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# --- AWS / Braket configuration ---
AWS_REGION = "us-west-2"
S3_BUCKET = "quantum-clinical-optimization-us-west-2"
S3_PREFIX_QUBO = "qubo/scenario_A"
S3_PREFIX_RESULTS = "results/braket/scenario_A"

# --- Local outputs ---
PATH_BRK_SUMMARY = RESULTS_DIR / "scenario_A_braket_sv1_summary.csv"
PATH_BRK_SELECTED = RESULTS_DIR / "scenario_A_braket_sv1_selected_trials.csv"
PATH_BRK_JOB_META = RESULTS_DIR / "scenario_A_braket_sv1_job_metadata.json"

print("OK: Directories ready.")
print("RESULTS_DIR:", RESULTS_DIR.resolve())
print("FIG_DIR:", FIG_DIR.resolve())
print("AWS_REGION:", AWS_REGION)
print("S3_BUCKET:", S3_BUCKET)


OK: Directories ready.
RESULTS_DIR: /home/parallels/projects/quantum-clinical-trial-optimization/data/results
FIG_DIR: /home/parallels/projects/quantum-clinical-trial-optimization/outputs/figures
AWS_REGION: us-west-2
S3_BUCKET: quantum-clinical-optimization-us-west-2


### What Cell 1 Just Did

- Imported core dependencies (NumPy/Pandas/boto3) and the AWS Braket SDK.
- Defined local output directories and filenames for Braket artifacts.
- Set canonical AWS region and S3 prefixes so inputs/outputs remain auditable and consistent.


In [4]:
# ============================================================
# Cell 2 — AWS / S3 sanity checks + Braket session + SV1 device (FIXED)
# ============================================================

import boto3
from braket.aws import AwsDevice, AwsSession

AWS_REGION = AWS_REGION  # keep from Cell 1

# --- boto3 session for STS/S3 sanity checks ---
boto_sess = boto3.Session(region_name=AWS_REGION)

sts = boto_sess.client("sts")
s3 = boto_sess.client("s3")

caller = sts.get_caller_identity()
print("AWS caller identity:")
print("  Account:", caller.get("Account"))
print("  ARN:", caller.get("Arn"))
print("  Region:", AWS_REGION)

# Bucket existence / access check
try:
    s3.head_bucket(Bucket=S3_BUCKET)
    print(f"OK: S3 bucket accessible: s3://{S3_BUCKET}/")
except Exception as e:
    raise PermissionError(
        f"Cannot access bucket s3://{S3_BUCKET}/ in region {AWS_REGION}.\n"
        "Fix: check AWS credentials, permissions (s3:ListBucket, s3:GetObject, s3:PutObject), and region.\n"
        f"Underlying error: {e}"
    )

# --- Braket session (this is what AwsDevice expects) ---
braket_sess = AwsSession(boto_session=boto_sess)

# Braket device ARN (SV1)
SV1_ARN = "arn:aws:braket:::device/quantum-simulator/amazon/sv1"
device = AwsDevice(SV1_ARN, aws_session=braket_sess)

print("Braket session region:", braket_sess.region)
print("Braket device name:", device.name)
print("Braket device ARN:", SV1_ARN)


AWS caller identity:
  Account: 581610642254
  ARN: arn:aws:iam::581610642254:user/jupyter-s3-access
  Region: us-west-2
OK: S3 bucket accessible: s3://quantum-clinical-optimization-us-west-2/
Braket session region: us-west-2
Braket device name: SV1
Braket device ARN: arn:aws:braket:::device/quantum-simulator/amazon/sv1


### What Cell 2 Just Did 

- Kept the same AWS identity + S3 bucket access verification using a standard `boto3.Session`.
- Created a **Braket-native `AwsSession`** (required by the Braket SDK) using that boto3 session.
- Initialized the **SV1** device using the correct session type so downstream `device.run(...)` calls work.


In [8]:
# ============================================================
# Cell 3 — Locate/load Scenario A QUBO + index→trial-id mapping (MAX ROBUST)
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ---------- File discovery ----------
def find_files(patterns, roots):
    hits = []
    for r in roots:
        r = Path(r)
        if not r.exists():
            continue
        for pat in patterns:
            hits.extend(r.rglob(pat))
    hits = [p for p in hits if p.is_file()]
    # stable ordering: shorter path first, then lexicographic
    return sorted(set(hits), key=lambda p: (len(p.parts), str(p).lower()))

roots = [Path("data/qubo_scenarios"), Path("data/results"), Path("data")]
patterns = [
    "*scenario_A*qubo*.json",
    "*scenario_A*QUBO*.json",
    "*scenario_A*qubo*.npz",
    "*scenario_A*qubo*.npy",
    "*scenario_A*qubo*.csv",
]

qubo_files = find_files(patterns, roots)
if not qubo_files:
    raise FileNotFoundError(
        "Could not find any Scenario A QUBO file in data/ (json/npz/npy/csv).\n"
        "Expected something like data/qubo_scenarios/scenario_A_qubo.json"
    )

PATH_QUBO = qubo_files[0]
print("Candidate QUBO files (top 10):")
for p in qubo_files[:10]:
    print("  -", p)
print("\nUsing:", PATH_QUBO)

# ---------- Helpers ----------
def is_number(x):
    # exclude booleans (they are ints)
    if isinstance(x, (bool, np.bool_)):
        return False
    return isinstance(x, (int, float, np.integer, np.floating))

def looks_like_square_matrix(lst, min_n=2, max_n=5000):
    if not isinstance(lst, list) or len(lst) < min_n:
        return False, None
    n = len(lst)
    if n > max_n:
        return False, None
    # must be list-of-lists and square-ish
    if not all(isinstance(row, list) for row in lst):
        return False, None
    if not all(len(row) == n for row in lst):
        return False, None
    # numeric check on a sample
    sample_ok = 0
    sample_total = 0
    step_i = max(1, n // 20)
    step_j = max(1, n // 20)
    for i in range(0, n, step_i):
        row = lst[i]
        for j in range(0, n, step_j):
            sample_total += 1
            if is_number(row[j]) or (row[j] is None):
                sample_ok += 1
    return (sample_ok / max(1, sample_total) >= 0.9), n

def strip_pair_key(k):
    s = str(k).strip()
    for ch in "()[]{}":
        s = s.replace(ch, "")
    s = s.replace(" ", "")
    return s

def parse_sparse_dict_to_Q(d, n_hint=None):
    # keys like "i,j" or "(i,j)"
    pairs = []
    for k, v in d.items():
        if not is_number(v):
            continue
        s = strip_pair_key(k)
        if "," not in s:
            continue
        a, b = s.split(",", 1)
        try:
            i, j = int(a), int(b)
            pairs.append((i, j))
        except Exception:
            continue
    if not pairs:
        return None
    n = max(max(i, j) for i, j in pairs) + 1
    if n_hint is not None:
        n = max(n, int(n_hint))
    Q = np.zeros((n, n), dtype=float)
    for k, v in d.items():
        if not is_number(v):
            continue
        s = strip_pair_key(k)
        if "," not in s:
            continue
        a, b = s.split(",", 1)
        try:
            i, j = int(a), int(b)
        except Exception:
            continue
        Q[i, j] += float(v)
    return Q

def parse_term_list_to_Q(terms, n_hint=None):
    # list of dict terms: i/j/value (or u/v/w, row/col/coef)
    idxs = []
    parsed = []
    for t in terms:
        if not isinstance(t, dict):
            continue
        i = t.get("i", t.get("u", t.get("row")))
        j = t.get("j", t.get("v", t.get("col")))
        v = t.get("value", t.get("w", t.get("coef", t.get("c"))))
        if i is None or j is None or v is None:
            continue
        try:
            i = int(i); j = int(j); v = float(v)
        except Exception:
            continue
        parsed.append((i, j, v))
        idxs.extend([i, j])

    if not parsed:
        return None
    n = max(idxs) + 1
    if n_hint is not None:
        n = max(n, int(n_hint))
    Q = np.zeros((n, n), dtype=float)
    for i, j, v in parsed:
        Q[i, j] += v
    return Q

def recursive_scan(obj, path="root", max_nodes=200000):
    """
    Recursively scan for:
      - square matrix candidates (list-of-lists numeric)
      - sparse dict candidates (keys like i,j numeric values)
      - list-of-terms candidates (list of dicts with i/j/value)
      - nct_id list candidates (strings starting with NCT)
    """
    stack = [(obj, path)]
    seen = 0

    matrices = []     # (path, n, list_obj)
    sparse_dicts = [] # (path, dict_obj)
    term_lists = []   # (path, list_obj)
    nct_lists = []    # (path, list_obj)

    while stack:
        cur, p = stack.pop()
        seen += 1
        if seen > max_nodes:
            break

        if isinstance(cur, list):
            # nct list candidate
            if cur and all(isinstance(x, str) for x in cur[: min(len(cur), 50)]):
                # heuristic: many entries like NCTxxxxx
                prefix_hits = sum(1 for x in cur[: min(len(cur), 200)] if str(x).startswith("NCT"))
                if prefix_hits >= max(5, int(0.5 * min(len(cur), 200))):
                    nct_lists.append((p, cur))

            ok, n = looks_like_square_matrix(cur, min_n=2, max_n=6000)
            if ok:
                matrices.append((p, n, cur))

            # term list candidate: list of dicts with i/j-ish keys
            if cur and isinstance(cur[0], dict):
                keys = set(cur[0].keys())
                if ({"i", "j"} & keys) and ({"value", "coef", "w", "c"} & keys):
                    term_lists.append((p, cur))

            # recurse elements (bounded)
            for idx, child in enumerate(cur[:5000]):  # don’t go insane
                stack.append((child, f"{p}[{idx}]"))

        elif isinstance(cur, dict):
            # sparse dict candidate?
            # heuristic: some keys contain comma and values numeric
            comma_keys = 0
            numeric_vals = 0
            for k, v in list(cur.items())[:5000]:
                if "," in strip_pair_key(k):
                    comma_keys += 1
                if is_number(v):
                    numeric_vals += 1
            if comma_keys >= 3 and numeric_vals >= 3:
                sparse_dicts.append((p, cur))

            for k, v in list(cur.items())[:5000]:
                stack.append((v, f"{p}.{k}"))

    return matrices, sparse_dicts, term_lists, nct_lists, seen

# ---------- Load QUBO file ----------
ext = PATH_QUBO.suffix.lower()
qubo_obj = None
Q = None
n = None
nct_ids = None
source_note = None

if ext == ".npz":
    npz = np.load(PATH_QUBO, allow_pickle=True)
    keys = list(npz.keys())
    print("Loaded NPZ keys:", keys)
    # pick best candidate array
    best_key = None
    best_n = -1
    for k in keys:
        arr = npz[k]
        if isinstance(arr, np.ndarray) and arr.ndim == 2 and arr.shape[0] == arr.shape[1]:
            if arr.shape[0] > best_n:
                best_n = arr.shape[0]
                best_key = k
    if best_key is None:
        raise ValueError(f"NPZ did not contain a square matrix array. Keys were: {keys}")
    Q = np.array(npz[best_key], dtype=float)
    n = Q.shape[0]
    source_note = f"npz:{PATH_QUBO} key:{best_key}"

elif ext == ".npy":
    arr = np.load(PATH_QUBO, allow_pickle=True)
    if not (isinstance(arr, np.ndarray) and arr.ndim == 2 and arr.shape[0] == arr.shape[1]):
        raise ValueError(f"NPY is not a square 2D matrix: shape={getattr(arr,'shape',None)}")
    Q = np.array(arr, dtype=float)
    n = Q.shape[0]
    source_note = f"npy:{PATH_QUBO}"

elif ext == ".csv":
    df = pd.read_csv(PATH_QUBO)
    # Try a few common CSV encodings:
    # 1) dense matrix with no headers
    if df.shape[0] == df.shape[1]:
        Q = df.to_numpy(dtype=float)
        n = Q.shape[0]
        source_note = f"csv(dense):{PATH_QUBO}"
    else:
        # 2) sparse triples: i,j,value
        cols = set(df.columns)
        if {"i", "j", "value"} <= cols or {"row", "col", "value"} <= cols:
            ic = "i" if "i" in cols else "row"
            jc = "j" if "j" in cols else "col"
            vc = "value"
            n = int(max(df[ic].max(), df[jc].max())) + 1
            Q = np.zeros((n, n), dtype=float)
            for _, r in df.iterrows():
                Q[int(r[ic]), int(r[jc])] += float(r[vc])
            source_note = f"csv(sparse-triples):{PATH_QUBO}"
        else:
            raise ValueError(f"CSV format not recognized for QUBO. Columns: {list(df.columns)}")

else:
    # JSON (or unknown treated as JSON)
    with open(PATH_QUBO, "r") as f:
        qubo_obj = json.load(f)

    # scan for candidates
    matrices, sparse_dicts, term_lists, nct_lists, visited = recursive_scan(qubo_obj)
    print(f"Scanned JSON nodes: {visited}")
    print(f"Found matrix candidates: {len(matrices)} | sparse dict candidates: {len(sparse_dicts)} | term-list candidates: {len(term_lists)}")
    if nct_lists:
        # pick the longest NCT list
        nct_lists_sorted = sorted(nct_lists, key=lambda x: len(x[1]), reverse=True)
        nct_path, nct_ids_candidate = nct_lists_sorted[0]
        nct_ids = [str(x) for x in nct_ids_candidate]
        print(f"Found nct_ids list at {nct_path} (len={len(nct_ids)})")

    # choose best Q source:
    # preference order: matrix (largest) > sparse dict (largest inferred) > term list
    chosen = None

    if matrices:
        # pick the largest square matrix; if nct_ids exists, prefer closest match
        if nct_ids:
            matrices_sorted = sorted(matrices, key=lambda t: (abs(t[1] - len(nct_ids)), -t[1]))
        else:
            matrices_sorted = sorted(matrices, key=lambda t: (-t[1],))
        mp, mn, mlist = matrices_sorted[0]
        Q = np.array(mlist, dtype=float)
        n = Q.shape[0]
        source_note = f"json.matrix:{PATH_QUBO} at:{mp} n:{n}"
        chosen = "matrix"

    if Q is None and sparse_dicts:
        # choose sparse dict that gives biggest n (or closest to nct_ids)
        best = None
        for sp, d in sparse_dicts:
            Qcand = parse_sparse_dict_to_Q(d, n_hint=(len(nct_ids) if nct_ids else None))
            if Qcand is None:
                continue
            nn = Qcand.shape[0]
            score = (abs(nn - len(nct_ids)) if nct_ids else 0, -nn)
            if best is None or score < best[0]:
                best = (score, sp, Qcand)
        if best is not None:
            _, sp, Q = best
            n = Q.shape[0]
            source_note = f"json.sparse_dict:{PATH_QUBO} at:{sp} n:{n}"
            chosen = "sparse_dict"

    if Q is None and term_lists:
        best = None
        for tp, terms in term_lists:
            Qcand = parse_term_list_to_Q(terms, n_hint=(len(nct_ids) if nct_ids else None))
            if Qcand is None:
                continue
            nn = Qcand.shape[0]
            score = (abs(nn - len(nct_ids)) if nct_ids else 0, -nn)
            if best is None or score < best[0]:
                best = (score, tp, Qcand)
        if best is not None:
            _, tp, Q = best
            n = Q.shape[0]
            source_note = f"json.term_list:{PATH_QUBO} at:{tp} n:{n}"
            chosen = "term_list"

    if Q is None:
        # Don’t silently continue—we need a QUBO matrix to run Braket
        # But dump useful debug info to the output so we can support the exact schema.
        preview = json.dumps(qubo_obj, indent=2)[:3000]
        raise ValueError(
            "Could not extract a QUBO matrix from the Scenario A QUBO JSON.\n\n"
            f"File: {PATH_QUBO}\n"
            f"Candidates found -> matrices:{len(matrices)}, sparse_dicts:{len(sparse_dicts)}, term_lists:{len(term_lists)}\n\n"
            "JSON preview (first ~3000 chars):\n"
            f"{preview}"
        )

print("\nQUBO loaded successfully.")
print("Source:", source_note)
print("Q shape:", Q.shape)
print("Q[0:3,0:3]:\n", Q[:3, :3])

# ---------- Ensure we have nct_ids for decoding ----------
if not nct_ids:
    # fallback: look for Scenario A candidates CSV
    cand_hits = find_files(["*scenario_A*candidate*.csv", "*scenario_A*candidates*.csv"], [Path("data/processed"), Path("data/results"), Path("data")])
    PATH_CAND = cand_hits[0] if cand_hits else None
    if PATH_CAND is not None:
        cand_df = pd.read_csv(PATH_CAND)
        if "nct_id" in cand_df.columns:
            nct_ids = cand_df["nct_id"].astype(str).tolist()
            print("Using candidates CSV ordering for nct_ids:", PATH_CAND, "len=", len(nct_ids))

# If still missing or length mismatch, pad/truncate to n
if not nct_ids:
    nct_ids = [f"unknown_{i}" for i in range(Q.shape[0])]
    print("WARN: No nct_ids mapping found; using placeholder IDs unknown_i.")

n = Q.shape[0]
if len(nct_ids) != n:
    if len(nct_ids) > n:
        nct_ids = nct_ids[:n]
    else:
        nct_ids = nct_ids + [f"unknown_{i}" for i in range(len(nct_ids), n)]
    print("Aligned nct_ids to Q dimension. nct_ids len:", len(nct_ids), "n:", n)


Candidate QUBO files (top 10):
  - data/qubo/scenario_A_qubo.json

Using: data/qubo/scenario_A_qubo.json
Scanned JSON nodes: 370
Found matrix candidates: 1 | sparse dict candidates: 0 | term-list candidates: 0
Found nct_ids list at root.nct_ids (len=18)

QUBO loaded successfully.
Source: json.matrix:data/qubo/scenario_A_qubo.json at:root.Q_dense n:18
Q shape: (18, 18)
Q[0:3,0:3]:
 [[-550.70588235  100.          100.        ]
 [ 100.         -550.70588235  100.        ]
 [ 100.          100.         -550.70588235]]


### What Cell 3 Just Did 

- Searched `data/` for Scenario A QUBO artifacts across **JSON / NPZ / NPY / CSV** formats.
- If the artifact is JSON, it performed a **recursive scan** to locate:
  - a square numeric Q matrix,
  - a sparse `(i,j) → value` dictionary,
  - or a list of weighted `(i,j,value)` terms,
  selecting the best candidate automatically.
- Extracted (or reconstructed) a deterministic `nct_ids` ordering for decoding bitstrings into trial IDs.
- Printed the exact internal path/format used to load Q so the run is reproducible.


In [9]:
# ============================================================
# Cell 4 — Convert QUBO (x in {0,1}) to Ising (s in {+1,-1}) terms
# ============================================================

import numpy as np

def qubo_to_ising(Q):
    """
    Convert QUBO: E(x)=x^T Q x, with x ∈ {0,1}^n
    to Ising:     E(s)=const + Σ h_i s_i + Σ_{i<j} J_ij s_i s_j, with s ∈ {+1,-1}^n
    using x_i = (1 - s_i)/2.

    Notes:
    - We symmetrize Q first (safe for energy evaluation and ZZ encoding stability).
    - Returned J is a dict keyed by (i,j) with i<j.
    """
    Q = np.array(Q, dtype=float)
    n = Q.shape[0]
    Qsym = 0.5 * (Q + Q.T)

    h = np.zeros(n, dtype=float)
    J = {}
    const = 0.0

    # Expand sum_{i,j} Q_ij x_i x_j with x_i=(1-s_i)/2
    # x_i x_j = (1 - s_i - s_j + s_i s_j)/4
    for i in range(n):
        for j in range(n):
            q = Qsym[i, j]
            if q == 0.0:
                continue

            const += q * 0.25
            h[i] += q * (-0.25)
            h[j] += q * (-0.25)

            if i == j:
                # s_i*s_i = 1 adds constant
                const += q * 0.25
            else:
                a, b = (i, j) if i < j else (j, i)
                J[(a, b)] = J.get((a, b), 0.0) + q * 0.25

    # Correct double-counting due to iterating all i,j
    h *= 0.5
    const *= 0.5
    for k in list(J.keys()):
        J[k] *= 0.5

    return h, J, const

h, J, const = qubo_to_ising(Q)

print("Ising conversion complete.")
print("  n qubits:", len(h))
print("  # couplers:", len(J))
print("  const:", float(const))

# Quick coefficient diagnostics
print("  h range:", float(np.min(h)), "to", float(np.max(h)))
if J:
    jvals = np.array(list(J.values()), dtype=float)
    print("  J range:", float(np.min(jvals)), "to", float(np.max(jvals)))

# Light sanity check: if a QUBO is all zeros, Ising is degenerate
if np.allclose(Q, 0.0):
    raise ValueError("QUBO matrix is all zeros. Cannot run meaningful QAOA.")

# Optional: warn on very large magnitudes (can make angles huge)
max_abs = max(np.max(np.abs(h)), np.max(np.abs(list(J.values())) if J else [0.0]))
if max_abs > 50:
    print("WARN: Very large Ising coefficients detected (|coeff| > 50).")
    print("      Consider scaling Q (or rescaling gamma/beta) to keep rotation angles reasonable.")


Ising conversion complete.
  n qubits: 18
  # couplers: 153
  const: 1346.8235294117649
  h range: -287.3235294117647 to -287.32352941176464
  J range: 25.0 to 25.0
WARN: Very large Ising coefficients detected (|coeff| > 50).
      Consider scaling Q (or rescaling gamma/beta) to keep rotation angles reasonable.


### What Cell 4 Just Did

- Converted the loaded Scenario A **QUBO matrix** into an equivalent **Ising model**:
  - `h` = single-qubit Z coefficients  
  - `J` = pairwise ZZ couplers  
  - `const` = energy offset (does not affect argmin)
- Reported coefficient ranges as a quick diagnostic (useful for spotting scaling issues before submitting Braket jobs).


In [11]:
# ============================================================
# Cell 5 — Build p=1 QAOA circuit + grid search on Braket SV1 (S3 DEST FIXED)
# ============================================================

import time
from braket.circuits import Circuit

# --- QAOA controls ---
P = 1
SHOTS = 200
GAMMA_GRID = np.linspace(0.2, 1.6, 5)
BETA_GRID  = np.linspace(0.2, 1.6, 5)

# Braket requires results to go to an amazon-braket-* bucket (default bucket is safest)
BRAKET_BUCKET = braket_sess.default_bucket()
BRAKET_TASK_PREFIX = f"{S3_PREFIX_RESULTS}/sv1_tasks"  # keeps project structure within Braket bucket
s3_destination_folder = (BRAKET_BUCKET, BRAKET_TASK_PREFIX)

print("Braket default bucket:", BRAKET_BUCKET)
print("Braket task results prefix:", f"s3://{BRAKET_BUCKET}/{BRAKET_TASK_PREFIX}/")

def qaoa_circuit_p1(h, J, gamma, beta):
    n = len(h)
    c = Circuit()
    for q in range(n):
        c.h(q)

    for i, hi in enumerate(h):
        if hi != 0.0:
            c.rz(i, 2.0 * float(gamma) * float(hi))

    for (i, j), Jij in J.items():
        if Jij == 0.0:
            continue
        c.cnot(i, j)
        c.rz(j, 2.0 * float(gamma) * float(Jij))
        c.cnot(i, j)

    for q in range(n):
        c.rx(q, 2.0 * float(beta))

    for q in range(n):
        c.measure(q)

    return c

def bitstring_to_x(bitstring, n):
    s = str(bitstring)
    if len(s) != n:
        s = s[:n].ljust(n, "0")
    return np.array([1 if ch == "1" else 0 for ch in s], dtype=int)

def qubo_energy(Q, x):
    x = np.asarray(x, dtype=float).reshape(-1, 1)
    return float((x.T @ Q @ x)[0, 0])

def run_counts_on_sv1(circuit, shots):
    task = device.run(
        circuit,
        shots=int(shots),
        s3_destination_folder=s3_destination_folder
    )
    result = task.result()
    return dict(result.measurement_counts), task

best = {
    "gamma": None,
    "beta": None,
    "energy": float("inf"),
    "bitstring": None,
    "counts": None,
    "task_id": None
}

print(f"Starting Braket SV1 grid search: {len(GAMMA_GRID)}x{len(BETA_GRID)} = {len(GAMMA_GRID)*len(BETA_GRID)} tasks")
print("Shots per task:", SHOTS)
t0 = time.time()

for gamma in GAMMA_GRID:
    for beta in BETA_GRID:
        circ = qaoa_circuit_p1(h, J, gamma=float(gamma), beta=float(beta))
        counts, task = run_counts_on_sv1(circ, SHOTS)

        local_best_E = float("inf")
        local_best_s = None
        for s, c in counts.items():
            x = bitstring_to_x(s, n=len(h))
            E = qubo_energy(Q, x)
            if E < local_best_E:
                local_best_E = E
                local_best_s = s

        if local_best_E < best["energy"]:
            best.update({
                "gamma": float(gamma),
                "beta": float(beta),
                "energy": float(local_best_E),
                "bitstring": str(local_best_s),
                "counts": counts,
                "task_id": getattr(task, "id", None) or getattr(task, "arn", None) or str(task)
            })

        print(f"gamma={gamma:.3f} beta={beta:.3f} | best_E_in_counts={local_best_E:.6f}")

elapsed_min = (time.time() - t0) / 60.0
print("\nBest result across grid:")
print(best)
print("Elapsed minutes:", elapsed_min)


Braket default bucket: amazon-braket-us-west-2-581610642254
Braket task results prefix: s3://amazon-braket-us-west-2-581610642254/results/braket/scenario_A/sv1_tasks/
Starting Braket SV1 grid search: 5x5 = 25 tasks
Shots per task: 200
gamma=0.200 beta=0.200 | best_E_in_counts=-1052.117647
gamma=0.200 beta=0.550 | best_E_in_counts=-1002.823529
gamma=0.200 beta=0.900 | best_E_in_counts=-1052.117647
gamma=0.200 beta=1.250 | best_E_in_counts=-1052.117647
gamma=0.200 beta=1.600 | best_E_in_counts=-1002.823529
gamma=0.550 beta=0.200 | best_E_in_counts=-1052.117647
gamma=0.550 beta=0.550 | best_E_in_counts=-1052.117647
gamma=0.550 beta=0.900 | best_E_in_counts=-1052.117647
gamma=0.550 beta=1.250 | best_E_in_counts=-1052.117647
gamma=0.550 beta=1.600 | best_E_in_counts=-901.411765
gamma=0.900 beta=0.200 | best_E_in_counts=-1052.117647
gamma=0.900 beta=0.550 | best_E_in_counts=-1052.117647
gamma=0.900 beta=0.900 | best_E_in_counts=-1002.823529
gamma=0.900 beta=1.250 | best_E_in_counts=-1052.117

### What Cell 5 Just Did 
- Built the p=1 QAOA circuit used fhe Braket results location.
- Braket tasks write outputs to the **Braket default bucket** (which starts with `amazon-braket-`), satisfying Braket validation requirements.
- Performed the (γ, β) grid search on SV1 and tracked the best observed QUBO energy solution.


In [12]:
# ============================================================
# Cell 6 — Decode best bitstring → selected trials + write artifacts + mirror to S3
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

# --- Defensive checks ---
if best.get("bitstring") is None:
    raise ValueError(
        "Cell 6 cannot run because Cell 5 did not produce a best bitstring.\n"
        "Fix: re-run Cell 5 and confirm at least one circuit completed successfully."
    )

def bitstring_to_x(bitstring, n):
    s = str(bitstring)
    if len(s) != n:
        s = s[:n].ljust(n, "0")
    return np.array([1 if ch == "1" else 0 for ch in s], dtype=int)

def qubo_energy(Q, x):
    x = np.asarray(x, dtype=float).reshape(-1, 1)
    return float((x.T @ Q @ x)[0, 0])

n = Q.shape[0]

# Ensure mapping exists and matches n
if not nct_ids:
    nct_ids = [f"unknown_{i}" for i in range(n)]
if len(nct_ids) != n:
    if len(nct_ids) > n:
        nct_ids = nct_ids[:n]
    else:
        nct_ids = nct_ids + [f"unknown_{i}" for i in range(len(nct_ids), n)]

# --- Decode best solution ---
best_x = bitstring_to_x(best["bitstring"], n=n)
selected_idx = np.where(best_x == 1)[0].tolist()
selected_nct_ids = [nct_ids[i] for i in selected_idx]

selected_df = pd.DataFrame({
    "index": selected_idx,
    "nct_id": selected_nct_ids
}).sort_values("index").reset_index(drop=True)

# --- Summarize run ---
summary = {
    "scenario": "A",
    "braket_device": getattr(device, "name", str(device)),
    "device_arn": SV1_ARN,
    "shots": int(SHOTS),
    "p": int(P),
    "gamma": float(best["gamma"]),
    "beta": float(best["beta"]),
    "best_observed_qubo_energy": float(best["energy"]),
    "selected_n": int(len(selected_idx)),
    "task_id": best.get("task_id"),
    "braket_results_bucket": str(BRAKET_BUCKET),
    "braket_results_prefix": str(BRAKET_TASK_PREFIX),
    "project_bucket": str(S3_BUCKET),
    "source_qubo_file": str(PATH_QUBO),
}

# --- Write local artifacts ---
pd.DataFrame([summary]).to_csv(PATH_BRK_SUMMARY, index=False)
selected_df.to_csv(PATH_BRK_SELECTED, index=False)

job_meta = {
    "summary": summary,
    "best_bitstring": best["bitstring"],
    "measurement_counts_top20": dict(sorted(best["counts"].items(), key=lambda kv: kv[1], reverse=True)[:20]),
}

with open(PATH_BRK_JOB_META, "w") as f:
    json.dump(job_meta, f, indent=2)

print("Wrote local artifacts:")
print("  -", PATH_BRK_SUMMARY)
print("  -", PATH_BRK_SELECTED)
print("  -", PATH_BRK_JOB_META)

# --- Build and write an input bundle for auditability ---
PATH_LOCAL_INPUT_BUNDLE = RESULTS_DIR / "scenario_A_braket_input_bundle.json"
with open(PATH_LOCAL_INPUT_BUNDLE, "w") as f:
    json.dump(
        {
            "source_qubo_path": str(PATH_QUBO),
            "n": int(n),
            "nct_ids": nct_ids,
            "qubo_matrix": np.array(Q, dtype=float).tolist(),
        },
        f,
        indent=2
    )

print("Wrote input bundle:", PATH_LOCAL_INPUT_BUNDLE)

# --- Upload inputs + outputs to your project bucket (OK even though tasks wrote to amazon-braket-*) ---
def s3_put_file(local_path: Path, bucket: str, key: str):
    s3.upload_file(str(local_path), bucket, key)
    return f"s3://{bucket}/{key}"

uploads = {}
uploads["input_bundle"] = s3_put_file(PATH_LOCAL_INPUT_BUNDLE, S3_BUCKET, f"{S3_PREFIX_QUBO}/scenario_A_braket_input_bundle.json")
uploads["summary_csv"]  = s3_put_file(PATH_BRK_SUMMARY,  S3_BUCKET, f"{S3_PREFIX_RESULTS}/scenario_A_braket_sv1_summary.csv")
uploads["selected_csv"] = s3_put_file(PATH_BRK_SELECTED, S3_BUCKET, f"{S3_PREFIX_RESULTS}/scenario_A_braket_sv1_selected_trials.csv")
uploads["job_meta"]     = s3_put_file(PATH_BRK_JOB_META, S3_BUCKET, f"{S3_PREFIX_RESULTS}/scenario_A_braket_sv1_job_metadata.json")

print("\nUploaded to project S3 bucket:")
for k, v in uploads.items():
    print(f"  - {k}: {v}")

# --- Quick peek ---
print("\nSelected trials (first 20):")
display(selected_df.head(20))

print("\nRun summary:")
display(pd.DataFrame([summary]))


Wrote local artifacts:
  - data/results/scenario_A_braket_sv1_summary.csv
  - data/results/scenario_A_braket_sv1_selected_trials.csv
  - data/results/scenario_A_braket_sv1_job_metadata.json
Wrote input bundle: data/results/scenario_A_braket_input_bundle.json

Uploaded to project S3 bucket:
  - input_bundle: s3://quantum-clinical-optimization-us-west-2/qubo/scenario_A/scenario_A_braket_input_bundle.json
  - summary_csv: s3://quantum-clinical-optimization-us-west-2/results/braket/scenario_A/scenario_A_braket_sv1_summary.csv
  - selected_csv: s3://quantum-clinical-optimization-us-west-2/results/braket/scenario_A/scenario_A_braket_sv1_selected_trials.csv
  - job_meta: s3://quantum-clinical-optimization-us-west-2/results/braket/scenario_A/scenario_A_braket_sv1_job_metadata.json

Selected trials (first 20):


,index,nct_id
0,6,NCT05821166
1,14,NCT05816655
2,16,NCT05815303



Run summary:


,scenario,braket_device,device_arn,shots,p,gamma,beta,best_observed_qubo_energy,selected_n,task_id,braket_results_bucket,braket_results_prefix,project_bucket,source_qubo_file
0,A,SV1,arn:aws:braket:::device/quantum-simulator/amaz...,200,1,0.2,0.2,-1052.117647,3,arn:aws:braket:us-west-2:581610642254:quantum-...,amazon-braket-us-west-2-581610642254,results/braket/scenario_A/sv1_tasks,quantum-clinical-optimization-us-west-2,data/qubo/scenario_A_qubo.json


### What Cell 6 Just Did

- Decoded the best Braket-sampled bitstring into selected indices and mapped those indices back to **NCT IDs** using the deterministic `nct_ids` ordering produced in Cell 3.
- Wrote portfolio-grade outputs locally:
  - `data/results/scenario_A_braket_sv1_summary.csv`
  - `data/results/scenario_A_braket_sv1_selected_trials.csv`
  - `data/results/scenario_A_braket_sv1_job_metadata.json`
- Created and saved an **input bundle** (`scenario_A_braket_input_bundle.json`) containing the QUBO matrix and index map so the run is auditable and reproducible.
- Uploaded the input bundle and output artifacts to your **project bucket** (`quantum-clinical-optimization-us-west-2`).  
  (Braket task raw outputs still live in the `amazon-braket-*` bucket, which is required by Braket.)

## Summary

We now have a complete, auditable AWS quantum execution for Scenario A:

- Braket SV1 ran QAOA tasks and stored raw task outputs in the required `amazon-braket-*` bucket.
- We decoded the best measured solution back into selected trial IDs.
- We saved and uploaded clean “portfolio artifacts” (summary, selections, metadata, plus the exact QUBO+mapping input bundle) into the project S3 bucket under stable prefixes.
